# 现代循环神经网络

## 门控循环单元(GRU)

### 门控隐状态

⻔控循环单元与普通的循环神经⽹络之间的关键区别在于：前者⽀持隐状态的⻔控。这意味着模型有专⻔的机制来确定应该何时更新隐状态，以及应该何时重置隐状态。这些机制是可学习的，并且能够解决了上⾯列出的问题。例如，如果第⼀个词元⾮常重要，模型将学会在第⼀次观测之后不更新隐状态。同样，模型也可以学会跳过不相关的临时观测。最后，模型还将学会在需要的时候重置隐状态。

#### 重置门和更新门

我们⾸先介绍重置⻔（resetgate）和更新⻔（updategate）。我们把它们设计成区间中的向量，这样我们就可以进⾏凸组合。重置⻔允许我们控制“可能还想记住”的过去状态的数量；更新⻔将允许我们控制新状态中有多少个是旧状态的副本。

图描述了⻔控循环单元中的重置⻔和更新⻔的输⼊，输⼊是由当前时间步的输⼊和前⼀时间步的隐状态给出。两个⻔的输出是由使⽤sigmoid激活函数的两个全连接层给出。

<div align="center">
    <img src="../img/在⻔控循环单元模型中计算重置⻔和更新⻔.png">
<div>

对于给定的时间步$t$，假设输⼊是⼀个⼩批量$X_t \in \mathbb{R}^{n \times d}$(样本个数，输⼊个数)，上⼀个时间步的隐状态是$H_{t-1} \in \mathbb{R}^{n \times h}$(隐藏单元个数)。那么，重置⻔$R_t \in \mathbb{R}^{n \times h}$和更新⻔$Z_t \in \mathbb{R}^{n \times h}$的计算如下所⽰：

$$
R_t = \sigma (X_t W_{xr} + H_{t - 1} W_{hr} + b_r)
$$
$$
Z_t = \sigma(X_t W_{xz} + H_{t - 1} W_{hz} + b_z)
$$

其中$W_{xr},W_{xz} \in \mathbb{R}^{d \times h}$和$W_{hr},W_{hz} \in \mathbb{R}^{h \times h}$是权重参数，$b_r,b_z \in \mathbb{R}^{1 \times h}$是偏置参数。请注意，在求和过程中会触发⼴播机制。我们使⽤sigmoid函数将输⼊值转换到区间(0, 1)。

#### 候选隐状态

在时间步$t$的候选隐状态$\tilde{H}_t \in \mathbb{R}^{n \times h}$:

$$
\tilde{H}_t = tanh(X_t W_{xh} + (R_t \odot H_{t - 1}) W_{hh} + b_h)
$$

其中$W_{xh} \in \mathbb{R}^{d \times h}$和$W_{hh} \in \mathbb{R}^{h \times h}$是权重参数，$b_h \in \mathbb{R}^{1 \times h}$是偏置项，符号$\odot$是Hadamard积（按元素乘积）运算符。在这⾥，我们使⽤tanh⾮线性激活函数来确保候选隐状态中的值保持在区间(-1, 1)中。

与普通循环神经网络相⽐，公式中的$R_t$和$H_{t - 1}$的元素相乘可以减少以往状态的影响。每当重置⻔$R_t$中的项接近$0$时，我们恢复⼀个如普通的循环神经⽹络。对于重置⻔$R_t$中所有接近$0$的项，候选隐状态是以$X_t$作为输⼊的多层感知机的结果。因此，任何预先存在的隐状态都会被重置为默认值。

<div align="center">
    <img src="../img/在⻔控循环单元模型中计算候选隐状态.png">
<div>

#### 隐状态

上述的计算结果只是候选隐状态，我们仍然需要结合更新⻔$Z_t$的效果。这⼀步确定新的隐状态$H_t \in \mathbb{R}^{n \times h}$在多⼤程度上来⾃旧的状态$H_{t-1}$和新的候选状态$\tilde{H}_t$。更新⻔$Z_t$仅需要在$H_{t-1}$和$\tilde{H}_t$之间进⾏按元素的凸组合就可以实现这个⽬标。这就得出了⻔控循环单元的最终更新公式：

$$
H_t = Z_t \odot H_{t-1} + (1 - Z_t) \odot \tilde{H}_t
$$

每当更新⻔$Z_t$接近$1$时，模型就倾向只保留旧状态。此时，来⾃$X_t$的信息基本上被忽略，从⽽有效地跳过了依赖链条中的时间步。相反，当$Z_t$接近$0$时，新的隐状态$H_t$就会接近候选隐状态$\tilde{H}_t$。这些设计可以帮助我们处理循环神经⽹络中的梯度消失问题，并更好地捕获时间步距离很⻓的序列的依赖关系。例如，如果整个⼦序列的所有时间步的更新⻔都接近于$1$，则⽆论序列的⻓度如何，在序列起始时间步的旧隐状态都将很容易保留并传递到序列结束。

<div align="center">
    <img src="../img/计算⻔控循环单元模型中的隐状态.png">
<div>

总之，⻔控循环单元具有以下两个显著特征：
- 重置⻔有助于捕获序列中的短期依赖关系；
- 更新⻔有助于捕获序列中的⻓期依赖关系。

#### GRU 门控小结

两个门都经 $\sigma$ 压到 $(0,1)$，按元素控制信息流，开多大由网络学习。

##### 1. 重置门 $R_t$（写候选时用）

$$
\tilde{H}_t = \tanh\bigl(X_t W_{xh} + (R_t \odot H_{t-1}) W_{hh} + b_h\bigr)
$$

- 控制：**算候选隐状态时，旧状态 $H_{t-1}$ 参与多少**
- $R\to 0$：几乎忽略过去，主要看当前输入（重置）
- $R\to 1$：过去充分参与，更像普通 RNN
- 作用偏向：**短期依赖**——哪些过去信息现在还相关

##### 2. 更新门 $Z_t$（写新状态时用）

$$
H_t = Z_t \odot H_{t-1} + (1 - Z_t) \odot \tilde{H}_t
$$

- 控制：**新隐状态里，旧记忆留多少、候选采纳多少**（凸组合）
- $Z\to 1$：基本保持 $H_{t-1}$，跳过本步更新
- $Z\to 0$：基本换成 $\tilde{H}_t$，写入新内容
- 作用偏向：**长期依赖**——重要信息可跨很多步原样传递

##### 一句话区分

| 门 | 一句话 |
|----|--------|
| 重置门 | 用 $R_t \odot H_{t-1}$ 控制：计算 $\tilde{H}_t$ 时，旧隐状态各维保留多少。偏短期 |
| 更新门 | 用 $Z_t$ 做凸组合：新的 $H_t$ 里，有多少来自旧的 $H_{t-1}$，有多少来自 $\tilde{H}_t$。偏长期 |